# doorplate: a tour

Parse US addresses into components, USPS Publication 28 and RESO output, a match key and a slug. All addresses here are invented. This library formats and parses; it **does not validate** that an address exists or can receive mail.

Run it with `pip install doorplate pandas polars duckdb pyarrow jupyter` (the DataFrame and SQL sections need their library; or build from source: `maturin develop` in `crates/doorplate-py`). The notebook runs from the `examples/` directory or the repository root.

In [ ]:
from pathlib import Path
import doorplate as dp

DATA = Path("examples/data") if Path("examples/data").exists() else Path("data")
print("library", dp.__version__)

## One address

`parse` never raises on odd input; it returns a `ParsedAddress` with `warnings` for anything it is unsure of. Only text with no address at all raises `ParseError`.

In [ ]:
a = dp.parse("123 north main street apt 4, Miami FL 33101")
print(a.street_number, a.street_dir_prefix, a.street_name, a.street_suffix)
print(a.unit_type, a.unit_number, "|", a.city, a.state, a.zip5)
print("warnings:", a.warnings)
a.to_dict()["place"]

## Output formats

The USPS form follows Publication 28; `to_reso()` uses RESO Data Dictionary 2.0 names. Both are computed when you ask.

In [ ]:
print(a.to_usps())
print(a.to_reso())

## Display and long forms

`to_usps()` is uppercase because Publication 28 asks for it. For showing an address to a person, `to_display()` gives mixed case with the same abbreviations and `to_long()` spells directionals, suffixes and unit types out. **Neither is a Pub 28 form**: use `to_usps()` for mail.


In [ ]:
print(a.to_display())
print(a.to_long())
print(a.to_display()["last_line"])


Ordinals, `PO Box` and highway names are cased by rule. The parse has already dropped apostrophes (`O'Brien` is `OBRIEN`), so a display form cannot restore them.


In [ ]:
for text in ["100 NE 5th Ave SW, Tampa FL 33601",
             "PO Box 12, Tampa FL 33601",
             "100 Kentucky State Highway 625",
             "289-01 Mid-Island Plz, Queens NY 11426"]:
    p = dp.parse(text)
    print(f"{p.to_usps()['line1']!s:32} | {p.to_display()['line1']!s:28} | {p.to_long()['line1']}")


## Keys and slugs

The **key** identifies the property (`k3:` prefix; the format is a contract); `key(unit=True)` adds the unit. The **slug** is URL-safe text.

In [ ]:
print(a.key())
print(a.key(unit=True))
print(a.slug())

# Different spellings, same key:
same = [dp.parse(t).key() for t in ["123 N Main St Apt 4, Miami FL 33101",
                                     "123 north main street #4 miami, fl 33101-2200"]]
print(same, same[0] == same[1])

## Messy lines, and what the warnings say

In [ ]:
for line in (DATA / "addresses_messy.txt").read_text().splitlines():
    p = dp.parse(line)
    print(f"{line[:46]:46} -> {p.to_usps()['line1']!s:28} {p.address_type():8} {p.warnings}")

`address_type()` is one of `street`, `po_box`, `rural_route`, `military`, `general_delivery`, `dual`, `unknown`. Text it cannot place is in `unparsed`:

In [ ]:
p = dp.parse("Main St & 1st Ave, Tampa FL")
print(p.warnings, repr(p.unparsed))

## Rare shapes

PO boxes, rural and highway-contract routes, military addresses and general delivery are shapes of their own, with structured fields and a key that starts with their type letter (`P`, `R`, `M`, `G`; a street is `S`). A street and a PO box on one line is `dual`. A highway keeps its prefix and route.

In [ ]:
for text in ["PO Box 12, Tampa FL 33601",
             "RR 2 Box 152, Springfield MO 65801",
             "HC 61 Box 7, Rolla MO 65401",
             "PSC 1650 Box 10, APO AE 09012",
             "General Delivery, Tampa FL 33601",
             "321 Elm Rd, PO Box 9, Dayton OH 45402",
             "100 State Route 9, Albany NY 12207"]:
    p = dp.parse(text)
    shape = {k: v for k, v in dict(po_box=p.po_box, rural_route=p.rural_route, military=p.military,
                                   post_office=p.military_post_office,
                                   highway=p.highway and p.highway["name"]).items() if v}
    print(f"{p.address_type():17} {p.key():40} {shape} {p.warnings}")

A unit can be followed by a second one (`second_unit_type`, `second_unit_number`), and an address with no ZIP still gets a key, with a weaker prefix (`k3n:`) and a warning:

In [ ]:
p = dp.parse("1100 Pennsylvania Ave NW Apt 5 Rm 2, Washington DC 20004")
print(p.unit_type, p.unit_number, "|", p.second_unit_type, p.second_unit_number)
p = dp.parse("12 elm st. #5 orlando fl")
print(p.key(), p.warnings)

## A state hint

`default_state` is a lookup hint for input with no state. It never fills `state`.

In [ ]:
print(dp.parse("281 Evelyn Dr Dothan").city)
p = dp.parse("281 Evelyn Dr Dothan", default_state="AL")
print(p.city, p.state, p.warnings)

## Place data

A national list of places and ZIPs finds the city on a line with no commas, and the place for a ZIP when the city is missing. `city` always keeps what the input said; `place` is what was found. See the [place data notes](../docs/places.md).

In [ ]:
for text in ["742 Evergreen Terrace Springfield 62704",   # no commas: the city is found
             "PALM BEACH, 33480",                          # a city and a ZIP
             "WINGATE ST, 32507"]:                         # a street and a ZIP, no city
    p = dp.parse(text)
    found = p.place and (p.place["name"], p.place["state"], p.place["source"])
    print(f"{text:42} city={p.city!s:12} place={found}")

## Fields already split

`parse_fields` takes address columns as you have them. A name-like first line goes to `attention`.

In [ ]:
p = dp.parse_fields(line1="THE GARDENS", line2="5120 PALM BEACH BLVD",
                     city="FORT MYERS", state="FL", zip="33905")
print("attention:", p.attention, "|", p.street_number, p.street_name, p.street_suffix)
print(p.slug())

## A batch

`parse_many` releases the GIL and parses in parallel. A row with no address text is `None`.

In [ ]:
import csv
with open(DATA / "addresses_oneline.csv", newline="") as f:
    rows = list(csv.DictReader(f))
parsed = dp.parse_many([r["address"] for r in rows])
for r, p in zip(rows, parsed):
    print(r["id"].rjust(2), p.key() if p else None)

## Pre-split columns, as a batch

`parse_fields_many` is `parse_many` for address columns: give it the columns (lists; an element may be `None`) and it returns one result per row, in order, with the GIL released. A row with no address text comes back `None`.


In [ ]:
with open(DATA / "addresses_split.csv", newline="") as f:
    split = list(csv.DictReader(f))
col = lambda name: [r[name] or None for r in split]   # an empty cell is None
split_parsed = dp.parse_fields_many(
    line1=col("line1"), line2=col("line2"), city=col("city"), state=col("state"), zip=col("zip"))
for r, p in zip(split, split_parsed):
    print(r["id"].rjust(2), p.to_display()["line1"] if p else None, "|", p.key() if p else None)


## pandas

`doorplate.pandas.parse_column` (and `.polars`) return component columns for a column of one-line addresses, with the same index and row order as the input. Install the extra: `pip install "doorplate[pandas]"`.

In [ ]:
import pandas as pd
from doorplate.pandas import parse_column

df = pd.read_csv(DATA / "addresses_oneline.csv")
out = df.join(parse_column(df["address"], columns=["street_name", "city", "state", "zip5", "key"]))
out[["id", "addr_street_name", "addr_city", "addr_state", "addr_zip5", "addr_key"]]

## Polars

`doorplate.polars.parse_column` does the same for a Polars column (`pip install "doorplate[polars]"`). Pick the columns you need: `warnings` is the codes joined by commas (empty when there are none), and a blank or missing address row is null in every new column.

In [ ]:
import polars as pl
from doorplate.polars import parse_column as pl_parse_column

pl_df = pl.read_csv(DATA / "addresses_oneline.csv")
cols = ["street_name", "city", "state", "zip5", "address_type", "key", "warnings"]
parsed_pl = pl_df.hstack(pl_parse_column(pl_df["address"], columns=cols))
parsed_pl.select("id", "addr_street_name", "addr_city", "addr_state", "addr_zip5", "addr_address_type", "addr_key")

Rows with a warning are the ones to review; rows with no address text at all stay in the frame with nulls:

In [ ]:
parsed_pl.filter(pl.col("addr_warnings") != "").select("id", "address", "addr_warnings")

The key is what makes two spellings of one property meet. Group on it to find duplicates, or join two tables on it:

In [ ]:
spellings = pl.DataFrame({"address": ["123 N Main St Apt 4, Miami FL 33101",
                                      "123 north main street #4 miami, fl 33101-2200",
                                      "9 Oak Ln, Tampa FL 33602"]})
spellings = spellings.hstack(pl_parse_column(spellings["address"], columns=["key"]))
spellings.group_by("addr_key", maintain_order=True).agg(pl.len().alias("rows"), pl.col("address"))

When your data already has the address in separate columns, give `parse_fields_many` the columns as lists (an empty cell is `None`) and add what you need back to the frame:

In [ ]:
split_pl = pl.read_csv(DATA / "addresses_split.csv", infer_schema_length=0)   # keep ZIP codes as text
rows_pl = dp.parse_fields_many(line1=split_pl["line1"].to_list(), line2=split_pl["line2"].to_list(),
                               city=split_pl["city"].to_list(), state=split_pl["state"].to_list(),
                               zip=split_pl["zip"].to_list())
split_pl.with_columns(
    key=pl.Series([p.key() if p else None for p in rows_pl]),
    usps=pl.Series([p.to_usps()["line1"] if p else None for p in rows_pl]),
).select("id", "line1", "line2", "usps", "key")

## DuckDB

`pip install "doorplate[duckdb]"` (DuckDB and PyArrow). Register two functions on a connection and use them in SQL, on a table, a CSV or a Parquet file. Each row is parsed once however many fields you read, in batches that Rust parses in parallel. Choose the columns you need when you register.

In [ ]:
import duckdb
from doorplate import duckdb as dd

con = duckdb.connect()
dd.register(con, columns=["street_name", "city", "state", "zip5", "address_type", "key", "warnings"])
oneline = DATA / "addresses_oneline.csv"
con.sql(f"""
    SELECT id,
           doorplate_parse(address).street_name AS street,
           doorplate_parse(address).city        AS city,
           doorplate_parse(address).key         AS key
    FROM read_csv('{oneline}')
""").df()

The rows to review, by warning:

In [ ]:
con.sql(f"""
    SELECT id, address, p.warnings
    FROM (SELECT *, doorplate_parse(address) AS p FROM read_csv('{oneline}'))
    WHERE p.warnings <> ''
""").df()

Separate columns go through `doorplate_parse_fields(line1, line2, city, state, zip)`. Here the key joins different spellings, and a count shows how many rows each property has:

In [ ]:
con.sql("""
    SELECT doorplate_parse(address).key AS key, count(*) AS rows, list(address) AS spellings
    FROM (VALUES ('123 N Main St Apt 4, Miami FL 33101'),
                 ('123 north main street #4 miami, fl 33101-2200'),
                 ('9 Oak Ln, Tampa FL 33602')) t(address)
    GROUP BY key ORDER BY key
""").df()

In [ ]:
split_csv = DATA / "addresses_split.csv"
con.sql(f"""
    SELECT id, doorplate_parse_fields(line1, line2, city, state, zip).key AS key
    FROM read_csv('{split_csv}', all_varchar=true)
""").df().head(4)

`parse_column` adds all the parsed columns to a relation and stays lazy, so it works on tables larger than memory. Pass `connection=` unless the relation came from `duckdb.sql(...)`:

In [ ]:
con.sql(f"CREATE OR REPLACE TABLE addresses AS SELECT * FROM read_csv('{oneline}')")
rel = dd.parse_column(con.table("addresses"), "address", connection=con)
rel.project("id, addr_street_name, addr_zip5, addr_key").limit(4).df()

## usaddress-style labels

If you come from `usaddress`, `parse_tokens` and `tag` give the same label names, and `address_type` is finer than usaddress's three types. Scoring and the differences are in the [compatibility notes](../docs/usaddress-compat.md).

In [ ]:
text = "123 Main St Apt 4, Miami FL 33101"
print(dp.parse_tokens(text))
print(dp.tag(text))

## The command line

The `doorplate` CLI does the same on a file: `doorplate parse-file IN.csv --column address -o OUT.csv` appends `addr_`-prefixed columns. Build it with `cargo build --release -p doorplate-cli` (the binary is `target/release/doorplate`).

`doorplate parse "<address>" --usps` prints the Pub 28 lines; `--display` and `--long` print the mixed-case and spelled-out forms (not Pub 28).


In [ ]:
import shutil, subprocess
exe = shutil.which("doorplate")
if exe is None:
    print("doorplate is not on PATH; build it with: cargo build --release -p doorplate-cli")
else:
    r = subprocess.run([exe, "parse-file", str(DATA / "addresses_oneline.csv"), "--column", "address"],
                       capture_output=True, text=True, check=True)
    print(r.stdout.splitlines()[0][:100], "...")
    print(r.stderr.strip())
    # The display and long forms are flags on `parse`:
    for flag in ("--display", "--long"):
        r = subprocess.run([exe, "parse", "123 north main street apt 4, miami fl 33101", flag],
                           capture_output=True, text=True, check=True)
        print(flag, "->", r.stdout.strip().replace("\n", " / "))
